# Recommendation checks and experiment planning

Run after notebook 04. This notebook independently checks cohort exposure, purchase timing and matched funnel periods using the verified shared dataset. It does not rerun unavailable business-team scripts, access an online report, establish causation or launch an experiment. See `docs/recommendation_validation.md` and `docs/experiment_protocols.md`.

Keep this source notebook unexecuted in Git. Save executed copies and generated results only outside every checkout.

In [ ]:
from pathlib import Path
import importlib.util
import json
import os
import sys

cwd = Path.cwd().resolve()
repo = next((p for p in [cwd, *cwd.parents] if (p / 'scripts' / 'check_recommendations.py').is_file()), None)
if repo is None:
    raise RuntimeError('Start Jupyter from this repository or its notebooks folder.')
sys.path.insert(0, str(repo / 'src'))
spec = importlib.util.spec_from_file_location('recommendation_cli', repo / 'scripts' / 'check_recommendations.py')
runner = importlib.util.module_from_spec(spec)
spec.loader.exec_module(runner)

report = runner.build_report(scenario_path=os.environ.get('WEAVVE_RECOMMENDATION_SCENARIO'))
checks = report['checks']
print('Verified dataset:', checks['dataset_id'])
print('Transaction cutoff:', checks['transaction_cutoff'])
print(report['evidence_status'])

## Definitions and observation limits

A customer-year-to-date average is not equal-age value. Current-extract order net may include refunds recorded later, so initial-order net is not automatically a historically available targeting signal. Use the matched-entry, fixed-follow-up output and read both the order-based and dated-ledger boundaries.

In [ ]:
print(json.dumps({key: checks[key] for key in ('definitions', 'limitations')}, indent=2, ensure_ascii=False))
print(json.dumps(checks['annual_cohorts'], indent=2, ensure_ascii=False))
print(json.dumps(checks['matched_followup_cohorts'], indent=2, ensure_ascii=False))

## Initial order, first calendar day and initial episode

The episode threshold is descriptive and learned from older mature cohorts. Compare all first-calendar-day orders before calling a later episode crossing day-one look-ahead. These historical summaries do not validate a prospective acquisition bidding rule.

In [ ]:
print(json.dumps(checks['initial_purchase_segments'], indent=2, ensure_ascii=False))

## Funnel periods

Compare the same complete months in each year. Monthly funnel and order-ledger populations are different. Any traffic/conversion decomposition is arithmetic, not a causal attribution or a guaranteed recoverable sales estimate.

In [ ]:
print(json.dumps(checks['matched_funnel'], indent=2, ensure_ascii=False))

## Optional conditional planning

Set `WEAVVE_RECOMMENDATION_SCENARIO` to a private JSON assumptions file before starting the kernel, or omit it to skip this section. Use the documented schema, include adverse and zero-lift cases, and list unpriced costs. Audience, margin and uplift remain assumptions until independently verified.

Binary-outcome power does not establish power for the primary contribution endpoint. Recruitment plus follow-up is different from recruitment alone. Mature cohort contribution is different from calendar-year realised profit. A changed allocation requires its own analysis plan.

In [ ]:
if report['planning_scenario'] is None:
    print('Skipped: no private planning assumptions supplied; no default profit forecast.')
else:
    print(json.dumps(report['planning_scenario'], indent=2, ensure_ascii=False))

## Save a private evidence record

The saved JSON records analysis code hashes and dataset identity. The writer rejects destinations within Git checkouts. Code checks do not complete the live checkout/CRM audit, establish a current eligible roster or approve either experiment.

In [ ]:
destination = runner.write_report(report)
print('Private report:', destination)